# Avaliação de Desempenho de um Modelo de Classificação de Coral-Sol(dataset anotado por Gabriela) e Análise de Explicabilidade com Grad-CAM
 
Autor:  Matheus de Farias Cavalcanti Santos
 
Atualizado: 06/03/2026
 
## Objetivo:

Este notebook tem como objetivo avaliar o desempenho de um modelo de classificação de Coral-Sol e analisar a coerência das explicações visuais geradas pelo Grad-CAM, com base nos dados registrados em planilha.

Para isso, são realizadas duas análises principais. A primeira consiste na comparação entre as colunas **LABEL** e **PREDIÇÃO**, com o propósito de medir a qualidade das previsões do modelo por meio da matriz de confusão e das métricas de **precision**, **recall**, **F1-score** e **acurácia**. Essa etapa permite verificar o quanto o modelo consegue classificar corretamente as amostras da base.

A segunda análise consiste na comparação entre a coluna **MODELO ACERTOU? SIM(1) NÃO(0)** e a coluna **GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?**, buscando investigar se existe concordância entre o acerto do modelo e a região destacada pelo Grad-CAM. Dessa forma, além de avaliar o desempenho preditivo, o notebook também busca analisar a interpretabilidade visual do modelo, verificando se as áreas de atenção destacadas fazem sentido em relação ao resultado obtido.

Assim, o notebook tem como finalidade fornecer uma avaliação quantitativa e interpretativa do modelo, unindo métricas de classificação e análise de explicabilidade visual para apoiar uma compreensão mais completa do seu comportamento.

🔗[Link para a planilha de análise das imagens classificação gradcam](https://petrobrasbr.sharepoint.com/:x:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/An%C3%A1lise%20das%20imagens%20classifica%C3%A7%C3%A3o%20gradcam.xlsx?d=wd5a7ebe833e6498f8fec726b48bc7141&csf=1&web=1&e=f9TFtc)

# Importações necessárias

In [9]:
from pathlib import Path
from typing import Dict, Tuple, List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

# Declaração de constantes

In [ ]:
ARQUIVO_EXCEL = r"[Diretório da planilha]"

# Funções necessárias

In [11]:
def carregar_planilha(caminho_arquivo: str) -> pd.DataFrame:
    """
    Carrega a planilha Excel em um DataFrame pandas.

    Parâmetros
    ----------
    caminho_arquivo : str
        Caminho do arquivo Excel.

    Retorno
    -------
    pd.DataFrame
        DataFrame contendo os dados da planilha.

    Exceções
    --------
    FileNotFoundError
        Caso o arquivo não seja encontrado.
    ValueError
        Caso a planilha esteja vazia.
    """
    caminho = Path(caminho_arquivo)

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")

    df = pd.read_excel(caminho)

    if df.empty:
        raise ValueError("A planilha foi carregada, mas está vazia.")

    return df


def validar_colunas(df: pd.DataFrame, colunas_obrigatorias: List[str]) -> None:
    """
    Verifica se todas as colunas obrigatórias existem no DataFrame.

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame a ser validado.
    colunas_obrigatorias : List[str]
        Lista com os nomes das colunas que devem existir.

    Exceções
    --------
    KeyError
        Caso alguma coluna obrigatória não exista.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]

    if colunas_ausentes:
        raise KeyError(
            "As seguintes colunas obrigatórias não foram encontradas na planilha: "
            f"{colunas_ausentes}"
        )


def padronizar_label_classificacao(valor: str) -> str:
    """
    Padroniza os valores de classificação textual.

    Exemplos:
    - 'positivo' -> 'POSITIVO'
    - ' NEGATIVO ' -> 'NEGATIVO'

    Parâmetros
    ----------
    valor : str
        Valor original da coluna.

    Retorno
    -------
    str
        Valor padronizado em caixa alta e sem espaços extras.
    """
    if pd.isna(valor):
        return np.nan

    return str(valor).strip().upper()


def padronizar_coluna_binaria(valor) -> float:
    """
    Padroniza colunas binárias para valores numéricos 0 ou 1.

    Aceita valores como:
    - 0, 1
    - '0', '1'
    - True, False

    Parâmetros
    ----------
    valor : any
        Valor original da coluna.

    Retorno
    -------
    float
        Retorna 0.0 ou 1.0.
        Retorna np.nan caso o valor não possa ser interpretado corretamente.
    """
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, bool):
        return float(int(valor))

    valor_str = str(valor).strip()

    if valor_str in {"0", "0.0"}:
        return 0.0
    if valor_str in {"1", "1.0"}:
        return 1.0

    try:
        valor_num = float(valor)
        if valor_num in (0.0, 1.0):
            return valor_num
    except Exception:
        pass

    return np.nan


def preparar_dados_classificacao(
    df: pd.DataFrame,
    coluna_real: str,
    coluna_predita: str,
    tarefa_binaria: bool = False
) -> pd.DataFrame:
    """
    Prepara os dados para análise de classificação.

    A função:
    - seleciona apenas as colunas necessárias;
    - remove linhas com valores ausentes;
    - padroniza os valores.

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame original.
    coluna_real : str
        Nome da coluna com o valor verdadeiro.
    coluna_predita : str
        Nome da coluna com o valor predito.
    tarefa_binaria : bool, opcional
        Se True, trata as colunas como binárias (0/1).
        Se False, trata as colunas como categóricas textuais.
        Padrão é False.

    Retorno
    -------
    pd.DataFrame
        DataFrame limpo e padronizado com as colunas:
        - 'y_true'
        - 'y_pred'
    """
    dados = df[[coluna_real, coluna_predita]].copy()
    dados.columns = ["y_true", "y_pred"]

    if tarefa_binaria:
        dados["y_true"] = dados["y_true"].apply(padronizar_coluna_binaria)
        dados["y_pred"] = dados["y_pred"].apply(padronizar_coluna_binaria)
    else:
        dados["y_true"] = dados["y_true"].apply(padronizar_label_classificacao)
        dados["y_pred"] = dados["y_pred"].apply(padronizar_label_classificacao)

    dados = dados.dropna(subset=["y_true", "y_pred"]).reset_index(drop=True)

    return dados


def calcular_metricas(
    y_true,
    y_pred,
    labels_ordenados: List,
    media: str = "binary",
    pos_label=None
) -> Dict[str, float]:
    """
    Calcula as métricas de classificação.

    Parâmetros
    ----------
    y_true : array-like
        Valores verdadeiros.
    y_pred : array-like
        Valores preditos.
    labels_ordenados : List
        Ordem dos labels a ser usada na matriz de confusão.
    media : str, opcional
        Tipo de média para métricas do sklearn.
        Exemplos: 'binary', 'macro', 'weighted'.
        Padrão é 'binary'.
    pos_label : any, opcional
        Classe positiva para tarefas binárias.
        Exemplo: 1 ou 'POSITIVO'.

    Retorno
    -------
    Dict[str, float]
        Dicionário com precision, recall, f1_score e acurácia.
    """
    metricas = {
        "precision": precision_score(
            y_true,
            y_pred,
            average=media,
            pos_label=pos_label,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            average=media,
            pos_label=pos_label,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_true,
            y_pred,
            average=media,
            pos_label=pos_label,
            zero_division=0
        ),
        "acuracia": accuracy_score(y_true, y_pred)
    }

    return metricas


def exibir_matriz_confusao(
    y_true,
    y_pred,
    labels_ordenados: List,
    titulo: str
) -> np.ndarray:
    """
    Gera e exibe a matriz de confusão.

    Parâmetros
    ----------
    y_true : array-like
        Valores verdadeiros.
    y_pred : array-like
        Valores preditos.
    labels_ordenados : List
        Ordem dos labels na matriz.
    titulo : str
        Título do gráfico.

    Retorno
    -------
    np.ndarray
        Matriz de confusão calculada.
    """
    matriz = confusion_matrix(y_true, y_pred, labels=labels_ordenados)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=labels_ordenados
    )
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()

    return matriz


def exibir_resultados(
    nome_analise: str,
    y_true,
    y_pred,
    labels_ordenados: List,
    media: str,
    pos_label=None
) -> None:
    """
    Exibe a matriz de confusão, métricas e relatório de classificação.

    Parâmetros
    ----------
    nome_analise : str
        Nome descritivo da análise.
    y_true : array-like
        Valores verdadeiros.
    y_pred : array-like
        Valores preditos.
    labels_ordenados : List
        Labels ordenados para a matriz de confusão.
    media : str
        Tipo de média usado nas métricas.
    pos_label : any, opcional
        Classe positiva para tarefas binárias.
    """
    print("=" * 80)
    print(f"ANÁLISE: {nome_analise}")
    print("=" * 80)

    matriz = exibir_matriz_confusao(
        y_true=y_true,
        y_pred=y_pred,
        labels_ordenados=labels_ordenados,
        titulo=f"Matriz de Confusão - {nome_analise}"
    )

    metricas = calcular_metricas(
        y_true=y_true,
        y_pred=y_pred,
        labels_ordenados=labels_ordenados,
        media=media,
        pos_label=pos_label
    )

    print("\nMatriz de confusão:")
    print(matriz)

    print("\nMétricas:")
    print(f"Precision : {metricas['precision']:.4f}")
    print(f"Recall    : {metricas['recall']:.4f}")
    print(f"F1-Score  : {metricas['f1_score']:.4f}")
    print(f"Acurácia  : {metricas['acuracia']:.4f}")

    print("\nRelatório de classificação:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels_ordenados,
            zero_division=0
        )
    )

    print("\n")

In [ ]:
# EXECUÇÃO PRINCIPAL

# Carrega os dados
df = carregar_planilha(ARQUIVO_EXCEL)

# Exibe as colunas para conferência
print("Colunas encontradas na planilha:")
for coluna in df.columns:
    print(f"- {coluna}")

# Valida se as colunas necessárias existem
colunas_obrigatorias = [
    "LABEL",
    "PREDIÇÃO",
    "MODELO ACERTOU? SIM(1) NÃO(0)",
    "GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?"
]
validar_colunas(df, colunas_obrigatorias)

## Análise 1 — Predição vs Label

Nesta análise, a coluna **PREDIÇÃO** é comparada com a coluna **LABEL**, que representa o rótulo verdadeiro de cada imagem.  
Essa é a análise principal de desempenho do modelo de classificação, pois permite verificar o quanto as previsões geradas pelo modelo estão de acordo com a verdade de referência presente na planilha.

Aqui, a interpretação é a seguinte:

- **LABEL**: classe verdadeira da amostra;
- **PREDIÇÃO**: classe prevista pelo modelo.

A partir dessa comparação, são calculadas a **matriz de confusão** e as métricas de **precision**, **recall**, **F1-score** e **acurácia**.

### O que essa análise mostra

Essa análise mostra o desempenho do modelo de classificação em distinguir corretamente as classes **POSITIVO(presença de Coral-Sol)** e **NEGATIVO(ausência de Coral-Sol)**.

- A **matriz de confusão** permite visualizar quantos exemplos foram classificados corretamente e quantos foram confundidos entre as classes.
- A **precision** indica, entre as amostras que o modelo previu como positivas, quantas realmente eram positivas.
- O **recall** indica, entre as amostras que eram realmente positivas, quantas o modelo conseguiu identificar corretamente.
- O **F1-score** representa um equilíbrio entre precision e recall.
- A **acurácia** mostra a proporção total de acertos do modelo considerando todas as amostras.

### Objetivo da análise

O objetivo desta etapa é avaliar, de forma quantitativa, a qualidade das previsões produzidas pelo modelo de classificação, utilizando como referência os rótulos reais da base.

In [ ]:
dados_predicao = preparar_dados_classificacao(
    df=df,
    coluna_real="LABEL",
    coluna_predita="PREDIÇÃO",
    tarefa_binaria=False
)

# Definição explícita da ordem das classes
labels_predicao = ["NEGATIVO", "POSITIVO"]

exibir_resultados(
    nome_analise="PREDIÇÃO vs LABEL",
    y_true=dados_predicao["y_true"],
    y_pred=dados_predicao["y_pred"],
    labels_ordenados=labels_predicao,
    media="binary",
    pos_label="POSITIVO"
)

## Conclusão — Desempenho do modelo (PREDIÇÃO vs LABEL)

Pela matriz de confusão, o modelo acertou **1302 NEGATIVOS (TN)** e **72 POSITIVOS (TP)**, porém cometeu **339 falsos positivos (FP)** e **122 falsos negativos (FN)**. Isso indica que ele tende a confundir uma parcela relevante dos exemplos, especialmente na classe **POSITIVO**.

As métricas reforçam esse comportamento:

- O desempenho para **NEGATIVO** é bom (**precision = 0.91**, **recall = 0.79**, **F1 = 0.85**), ou seja, o modelo consegue identificar a classe negativa com qualidade.
- Já para **POSITIVO**, o desempenho é fraco (**precision = 0.18**, **recall = 0.37**, **F1 = 0.24**). Em termos práticos:
  - Quando o modelo prevê **POSITIVO**, ele erra com frequência (muitos **falsos positivos**), pois a precisão é baixa.
  - Além disso, ele deixa passar muitos casos positivos reais (existem **122 FN**), pois o recall é baixo/moderado.

Apesar da **acurácia** ser **0.7488**, ela é influenciada pelo desbalanceamento do conjunto (muito mais amostras **NEGATIVAS** do que **POSITIVAS**). Por isso, a acurácia isoladamente pode dar uma impressão otimista, enquanto o **F1 da classe POSITIVO (0.24)** mostra que o modelo ainda não está confiável para identificar corretamente os positivos.

**Conclusão:** o modelo está adequado para reconhecer **NEGATIVOS**, mas apresenta **baixa capacidade de detecção de POSITIVOS**, com muitos erros tanto por *alarme falso* (FP) quanto por *não detectar positivos* (FN). Se a classe **POSITIVO** for a mais importante do problema, este desempenho ainda é insuficiente e precisa de melhorias (por exemplo: lidar com desbalanceamento, ajustar limiar de decisão e/ou otimizar o treinamento para aumentar recall/precision da classe positiva).

## Análise 2 — Grad-CAM vs Modelo Acertou

Nesta análise, a coluna **GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?** é comparada com a coluna **MODELO ACERTOU? SIM(1) NÃO(0)**.

Diferentemente da análise anterior, esta etapa **não avalia diretamente a classificação do modelo**, mas sim a relação entre o acerto do modelo e a coerência visual do mapa de ativação gerado pelo Grad-CAM.

Aqui, a interpretação é a seguinte:

- **MODELO ACERTOU? SIM(1) NÃO(0)**: indica se a predição do modelo estava correta ou incorreta;
- **GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?**: indica se o heatmap gerado pelo Grad-CAM destacou uma região da imagem que faz sentido do ponto de vista visual e semântico.

### O que essa análise mostra

Essa análise permite verificar se existe **concordância** entre o comportamento do modelo e a explicação visual produzida pelo Grad-CAM.

Em outras palavras, ela ajuda a responder perguntas como:

- Quando o modelo acerta, o Grad-CAM costuma focar na região correta?
- Quando o modelo erra, o Grad-CAM tende a destacar regiões incoerentes?
- Existe alinhamento entre a decisão do modelo e a área da imagem que recebeu maior atenção?

A partir dessa comparação, também são calculadas a **matriz de confusão** e as métricas de **precision**, **recall**, **F1-score** e **acurácia**.  
No entanto, nesta análise, essas métricas devem ser interpretadas como medidas de **concordância entre duas variáveis binárias**, e não como uma avaliação clássica de um classificador supervisionado.

### Objetivo da análise

O objetivo desta etapa é investigar se a explicação visual fornecida pelo Grad-CAM está consistente com o comportamento do modelo, ajudando na interpretação e na confiabilidade dos resultados obtidos.

In [ ]:
dados_gradcam = preparar_dados_classificacao(
    df=df,
    coluna_real="MODELO ACERTOU? SIM(1) NÃO(0)",
    coluna_predita="GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?",
    tarefa_binaria=True
)

# Converte para inteiro após limpeza
dados_gradcam["y_true"] = dados_gradcam["y_true"].astype(int)
dados_gradcam["y_pred"] = dados_gradcam["y_pred"].astype(int)

labels_gradcam = [0, 1]

exibir_resultados(
    nome_analise="GRADCAM vs MODELO ACERTOU",
    y_true=dados_gradcam["y_true"],
    y_pred=dados_gradcam["y_pred"],
    labels_ordenados=labels_gradcam,
    media="binary",
    pos_label=1
)

## Conclusão — GRADCAM vs MODELO ACERTOU (análise de concordância)

Nesta análise, a coluna **GRADCAM (0/1)** está sendo usada como um “indicador” de quando o modelo acertou (**MODELO ACERTOU = 1**) ou errou (**MODELO ACERTOU = 0**). Portanto, as métricas devem ser interpretadas como **concordância** entre “a atenção faz sentido” e “o modelo acertou”, e não como uma avaliação clássica de um classificador supervisionado.

Pela matriz de confusão:

- Quando o modelo **errou (0)**, o Grad-CAM indicou **atenção errada (0)** em **249** casos, mas indicou **atenção certa (1)** em **211** casos.
- Quando o modelo **acertou (1)**, o Grad-CAM indicou **atenção certa (1)** em **984** casos, porém indicou **atenção errada (0)** em **391** casos.

As métricas mostram um alinhamento **moderado**:

- **Precision = 0.8234**: quando o Grad-CAM foi marcado como **1 (atenção faz sentido)**, em geral o modelo estava correto (bom poder de confirmação).
- **Recall = 0.7156**: entre todos os acertos do modelo, cerca de 71,6% tiveram Grad-CAM marcado como **1** (o Grad-CAM “acompanha” muitos acertos, mas não todos).
- **F1-score = 0.7658**: indica um equilíbrio razoável entre precisão e cobertura nessa concordância.
- **Acurácia = 0.6719**: a concordância total é moderada, e deve ser lida com cuidado porque há desbalanceamento (muito mais casos com **MODELO ACERTOU = 1** do que **0**).

**Conclusão:** existe uma tendência de que, quando o Grad-CAM “faz sentido” (1), o modelo realmente tenha acertado — porém essa relação não é perfeita. Há um volume relevante de casos em que o modelo acerta, mas o Grad-CAM é avaliado como incoerente (**391**), e também casos em que o modelo erra, mas o Grad-CAM parece coerente (**211**). Assim, o Grad-CAM é um **sinal útil de interpretabilidade**, mas **não é suficiente sozinho** para garantir que a decisão do modelo está correta ou que a explicação visual está sempre alinhada com o resultado.

## Sugestões de melhorias para o modelo

Com base nas métricas obtidas, observa-se que o modelo apresenta bom desempenho para a classe **NEGATIVO**, mas ainda possui dificuldade em identificar corretamente a classe **POSITIVO**. Além disso, a análise envolvendo o Grad-CAM mostrou que a coerência visual das regiões de atenção ainda não está totalmente alinhada com os acertos do modelo. Dessa forma, algumas melhorias podem ser adotadas para aumentar tanto o desempenho preditivo quanto a confiabilidade das explicações visuais.

### 1. Tratar o desbalanceamento entre as classes

Uma das principais limitações observadas é o desbalanceamento da base, com predominância de exemplos da classe **NEGATIVO**. Isso pode levar o modelo a favorecer a classe majoritária durante o treinamento.

Algumas estratégias para reduzir esse problema são:

- aplicar **oversampling** da classe minoritária;
- aplicar **undersampling** da classe majoritária;
- utilizar **pesos de classe** na função de perda;
- empregar técnicas como **data augmentation** direcionadas à classe positiva.

Essas abordagens ajudam o modelo a aprender melhor os padrões da classe **POSITIVO**, aumentando a chance de melhorar recall e F1-score dessa classe.

### 2. Melhorar a qualidade e a representatividade dos dados

O desempenho de um modelo depende fortemente da qualidade da base de treinamento. Portanto, é importante verificar:

- se existem rótulos incorretos ou inconsistentes;
- se há imagens ambíguas ou de baixa qualidade;
- se a classe positiva possui exemplos suficientemente variados;
- se o conjunto de treino representa bem os diferentes cenários do problema.

Caso existam poucos exemplos positivos ou pouca diversidade nessa classe, o modelo tende a generalizar mal.

### 3. Aplicar mais técnicas de data augmentation

O uso de **data augmentation** pode tornar o modelo mais robusto e melhorar sua capacidade de generalização.

Alguns exemplos são:

- rotação;
- espelhamento;
- zoom;
- variação de brilho e contraste;
- pequenas translações;
- recortes controlados.

Essas transformações devem ser aplicadas com cuidado para não alterar características importantes da imagem. Quando bem utilizadas, podem ajudar o modelo a aprender padrões mais relevantes e menos dependentes de detalhes específicos do conjunto original.

### 4. Testar outras arquiteturas ou ajustes de hiperparâmetros

Outra melhoria importante é testar diferentes configurações do modelo, como:

- outras arquiteturas de rede neural;
- taxa de aprendizado;
- número de épocas;
- tamanho do batch;
- função de perda;
- otimizador;
- técnicas de regularização, como dropout.

Às vezes, o problema não está apenas nos dados, mas também na configuração do treinamento. Um ajuste mais refinado pode melhorar significativamente o desempenho.

### 5. Avaliar os erros de forma qualitativa

Além das métricas numéricas, é importante analisar visualmente:

- os **falsos positivos**;
- os **falsos negativos**;
- os casos em que o Grad-CAM destacou regiões incoerentes.

Essa inspeção pode revelar padrões de erro, como:
- confusão causada por ruído na imagem;
- regiões irrelevantes influenciando a decisão;
- inconsistência entre o objeto de interesse e a área destacada pelo modelo.

Essa análise pode orientar melhorias mais específicas na base ou na arquitetura.